# 201: Cumulative Net-Negative CO2 Emissions

This notebook calculates the cumulative net-negative CO2 emissions for each scenario that achieves net-zero CO2, run once per variant of the scenario data:

1. **base** - the real, as-submitted scenarios (`100_harmonised_infilled_data.csv`)
2. **sv_nzco2** - the synthetic NZCO2-hold stylised variant (`101_netzero_co2_extended.csv`)
3. **sv_nzghg** - the synthetic NZGHG-hold stylised variant (`101_netzero_kyoto_extended.csv`)

**Definition:** Cumulative net-negative CO2 emissions are the integral of CO2 emissions from the year a scenario reaches net-zero CO2 until 2100 (same reference year - `year_netzero_co2` - for all three variants; it's the CO2 *trajectory* that differs between them, not which year cumulation starts from).

CH4-decline terciles ("Low/Medium/High CH4 decline") are computed once from the base variant, then applied as fixed bins to the two stylised variants, so the labels stay comparable across all three outputs.

**Common input:** `101_netzero_timings.csv` - net-zero year metadata for each scenario

**Outputs:**
- `201_netzero_timings_cumnetnegco2.csv` (base)
- `201_netzero_timings_cumnetnegco2-svnzco2.csv` (sv_nzco2)
- `201_netzero_timings_cumnetnegco2-svnzghg.csv` (sv_nzghg)

In [1]:
import os
import dotenv
import pyam
import pandas as pd

from pathlib import Path

dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(


In [2]:
NETZERO_TIMINGS_FILE = OUTPUT_FOLDER / "101_netzero_timings.csv"

VARIANTS = [
    {
        "label": "base",
        "input_file": OUTPUT_FOLDER / "100_harmonised_infilled_data.csv",
        "scenario_suffix": None,
        "scenario_group": "Original scenarios",
        "output_file": OUTPUT_FOLDER / "201_netzero_timings_cumnetnegco2.csv",
    },
    {
        "label": "sv_nzco2",
        "input_file": OUTPUT_FOLDER / "101_netzero_co2_extended.csv",
        "scenario_suffix": "_NZCO2",
        "scenario_group": "Stylised Variants (NZCO2-hold)",
        "output_file": OUTPUT_FOLDER / "201_netzero_timings_cumnetnegco2-svnzco2.csv",
    },
    {
        "label": "sv_nzghg",
        "input_file": OUTPUT_FOLDER / "101_netzero_kyoto_extended.csv",
        "scenario_suffix": "_NZKyoto",
        "scenario_group": "Stylised Variants (NZGHG-hold)",
        "output_file": OUTPUT_FOLDER / "201_netzero_timings_cumnetnegco2-svnzghg.csv",
    },
]

In [3]:
def load_variant(config):
    """Load one variant's emissions data, strip its scenario-name suffix (if any) so
    scenario names match the original scenarios, tag it with its scenario_group, and
    merge in net-zero timing metadata."""
    df = pyam.IamDataFrame(config["input_file"])

    if config["scenario_suffix"] is not None:
        data = df.data.copy()
        data["scenario"] = data["scenario"].str.replace(
            config["scenario_suffix"], "", regex=False
        )
        df = pyam.IamDataFrame(data)

    df.load_meta(NETZERO_TIMINGS_FILE)
    df.set_meta(config["scenario_group"], name="scenario_group")

    # Match the exact scenario universe used everywhere else downstream
    # (300_prepare_scenarios.ipynb onward - MAGICC runs, 500-series metrics, 501-506):
    # achieve_netzero_co2 AND year_netzero_co2 <= 2090. achieve_netzero_co2 alone
    # (True for anything reaching net-zero by 2100) is a broader set that was never
    # actually run through MAGICC - mixing those ~80 extra scenarios in here skews
    # the CH4-decline terciles and breaks the cumulative_netnegative_co2 integration
    # below (their integration window collapses to almost nothing, since it starts
    # only a few years before 2100).
    df.set_meta(
        df.meta["achieve_netzero_co2"] & (df.meta["year_netzero_co2"] <= 2090),
        name="in_magicc_universe",
    )
    return df

In [4]:
CH4_DECLINE_LABELS = ["Low CH4 decline", "Medium CH4 decline", "High CH4 decline"]


def process_variant(config, ch4_bins=None):
    """Calculate cumulative net-negative CO2 (and CH4-decline terciles) for one
    variant, save the result, and return (df_co2, ch4_bins) - `ch4_bins` is derived
    fresh (via qcut) if not supplied, otherwise reused (via cut) so the tercile
    labels stay comparable across variants."""
    df = load_variant(config)

    # Calculate methane emissions reduction (2023 to 2043) for scenarios in the
    # MAGICC universe (achieve_netzero_co2 AND year_netzero_co2 <= 2090)
    df_ch4 = df.filter(variable="Emissions|CH4", in_magicc_universe=True).timeseries()
    ch4_decline = df_ch4.apply(lambda x: x[2023] - x[2043], axis=1).round(2)

    # Filter to the same universe and aggregate Fossil + Biosphere
    df_netzero = df.filter(in_magicc_universe=True)
    df_co2 = df_netzero.add(
        a="Emissions|CO2|Fossil", b="Emissions|CO2|Biosphere", name="Emissions|CO2"
    )
    df_co2.convert_unit(current="Mt CO2/yr", to="Gt CO2/yr", inplace=True)

    # Integrate CO2 emissions from each scenario's net-zero year to 2100
    cumulative_netnegative_co2 = df_co2.timeseries().apply(
        lambda x: pyam.timeseries.cumulative(
            x,
            first_year=df_co2.meta.loc[x.name[0:2], "year_netzero_co2"],
            last_year=2100,
        ).round(2),
        axis=1,
    )
    df_co2.set_meta(cumulative_netnegative_co2, name="cumulative_netnegative_co2")
    df_co2.set_meta(ch4_decline, name="ch4_decline")

    if ch4_bins is None:
        ch4_decline_groups, ch4_bins = pd.qcut(
            ch4_decline, q=3, labels=CH4_DECLINE_LABELS, retbins=True
        )
    else:
        ch4_decline_groups = pd.cut(
            ch4_decline, bins=ch4_bins, labels=CH4_DECLINE_LABELS, include_lowest=True
        )
    df_co2.set_meta(ch4_decline_groups, name="ch4_decline_group")

    print(f"\n=== {config['label']} ===")
    print("CH4 decline group counts:")
    print(ch4_decline_groups.value_counts().sort_index())
    print("\nCH4 decline ranges per group:")
    for group in CH4_DECLINE_LABELS:
        values = ch4_decline[ch4_decline_groups == group]
        if values.empty:
            print(f"  {group}: (no scenarios)")
            continue
        print(f"  {group}: {values.min():.1f} to {values.max():.1f} Mt CH4/yr")
    print(f"Unassigned scenarios: {ch4_decline_groups.isna().sum()}")

    df_co2.meta.to_csv(config["output_file"])
    print(f"Saved to {config['output_file']}")

    return df_co2, ch4_bins

In [5]:
# Process base first to derive the CH4-decline bin edges, then reuse them for the
# two stylised variants so "Low/Medium/High CH4 decline" is directly comparable
# across all three outputs.
ch4_bins = None
results = {}
for variant_config in VARIANTS:
    results[variant_config["label"]], ch4_bins = process_variant(
        variant_config, ch4_bins=ch4_bins
    )

[INFO] 13:29:29 - pyam.core: Reading file /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/100_harmonised_infilled_data.csv


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/pyam/core.py:933: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  self.meta[name] = meta[name].combine_first(self.meta[name])


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/pyam/core.py:933: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  self.meta[name] = meta[name].combine_first(self.meta[name])
[INFO] 13:29:34 - pyam.core: Reading file /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/101_netzero_co2_extended.csv



=== base ===
CH4 decline group counts:
Low CH4 decline       261
Medium CH4 decline    261
High CH4 decline      262
Name: count, dtype: int64

CH4 decline ranges per group:
  Low CH4 decline: -76.6 to 119.5 Mt CH4/yr
  Medium CH4 decline: 119.7 to 145.0 Mt CH4/yr
  High CH4 decline: 145.4 to 262.0 Mt CH4/yr
Unassigned scenarios: 0
Saved to /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/201_netzero_timings_cumnetnegco2.csv


[WARNING] 13:29:39 - pyam.core: Ignoring meta indicators for the following scenarios:
         model       scenario
0  AIM/CGE 2.0        SSP1-26
1  AIM/CGE 2.0        SSP1-34
2  AIM/CGE 2.0        SSP1-45
3  AIM/CGE 2.0  SSP1-Baseline
4  AIM/CGE 2.0        SSP2-34
...


...


=== sv_nzco2 ===
CH4 decline group counts:
Low CH4 decline       261
Medium CH4 decline    261
High CH4 decline      262
Name: count, dtype: int64

CH4 decline ranges per group:
  Low CH4 decline: -76.6 to 119.5 Mt CH4/yr
  Medium CH4 decline: 119.7 to 145.0 Mt CH4/yr
  High CH4 decline: 145.4 to 262.0 Mt CH4/yr
Unassigned scenarios: 0
Saved to /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/201_netzero_timings_cumnetnegco2-svnzco2.csv


[WARNING] 13:29:43 - pyam.core: Ignoring meta indicators for the following scenarios:
         model       scenario
0  AIM/CGE 2.0        SSP1-26
1  AIM/CGE 2.0        SSP1-34
2  AIM/CGE 2.0        SSP1-45
3  AIM/CGE 2.0  SSP1-Baseline
4  AIM/CGE 2.0        SSP2-26
...


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/pyam/core.py:933: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  self.meta[name] = meta[name].combine_first(self.meta[name])



=== sv_nzghg ===
CH4 decline group counts:
Low CH4 decline       127
Medium CH4 decline    119
High CH4 decline      141
Name: count, dtype: int64

CH4 decline ranges per group:
  Low CH4 decline: -76.6 to 119.0 Mt CH4/yr
  Medium CH4 decline: 119.7 to 145.0 Mt CH4/yr
  High CH4 decline: 145.4 to 262.0 Mt CH4/yr
Unassigned scenarios: 0
Saved to /Users/hoegner/GitHub/ar7_netzero_assessment/analysis/output_revision/201_netzero_timings_cumnetnegco2-svnzghg.csv


/Users/hoegner/GitHub/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/pyam/core.py:933: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  self.meta[name] = meta[name].combine_first(self.meta[name])
